In [1]:
from COSMOtherm.src.COSMOtherm_functions.Outputfile_Ingestion import list_files_with_extension, get_output_df
from COSMOtherm.Configfiles import COSMOthermConfig as COSMOConfig
from COSMOtherm.Configfiles import Config
from COSMOtherm.src.funcs import load_and_sort_solvents

solvents = load_and_sort_solvents(Config.solvents_fullpath)
filelist = list_files_with_extension(folder_path=COSMOConfig.odir+r"\CompleteScreening", file_extension="tab")

q:\Groups\eicr students\Stefan Tönnis\GIT_Repos\BayesianThompsonSamplingOptimization\COSMOtherm\src\funcs.py:51: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  solvents = pd.read_csv(solvents_fullpath, delim_whitespace=True)


In [2]:
output_data = get_output_df(files=filelist)


In [ ]:
failed_convergence_df = output_data[output_data['Warning msg'].str.strip() != '']
failed_convergence_df

,Temperature (°C),Phase 1 x(1) input,Phase 1 x(2) input,Phase 1 x(3) input,Phase 2 x(1) input,Phase 2 x(2) input,Phase 2 x(3) input,Compound 1,Compound 2,Compound 3,Phase 1 x(1) output,Phase 1 x(2) output,Phase 1 x(3) output,Phase 2 x(1) output,Phase 2 x(2) output,Phase 2 x(3) output,Warning msg
917,20.0,0.9669,0,0.0331,0,1.0,0,h2o,butanone,lacticacid,0.816263,0.171066,0.012671,0.463015,0.518552,0.018433,Forced Convergence in LIQ_EX - results might n...
918,20.0,0.9543,0,0.0457,0,1.0,0,h2o,butanone,lacticacid,0.802313,0.179754,0.017933,0.476011,0.498767,0.025221,Forced Convergence in LIQ_EX - results might n...
919,20.0,0.9407,0,0.0593,0,1.0,0,h2o,butanone,lacticacid,0.586283,0.384243,0.029474,0.512265,0.458117,0.029618,Forced Convergence in LIQ_EX - results might n...
925,30.0,0.9669,0,0.0331,0,1.0,0,h2o,butanone,lacticacid,0.849609,0.139040,0.011351,0.478187,0.503140,0.018673,Forced Convergence in LIQ_EX - results might n...
926,30.0,0.9543,0,0.0457,0,1.0,0,h2o,butanone,lacticacid,0.835038,0.148701,0.016261,0.491652,0.482851,0.025496,Forced Convergence in LIQ_EX - results might n...
927,30.0,0.9407,0,0.0593,0,1.0,0,h2o,butanone,lacticacid,0.561693,0.409025,0.029283,0.555860,0.414835,0.029306,Forced Convergence in LIQ_EX - results might n...
933,40.0,0.9669,0,0.0331,0,1.0,0,h2o,butanone,lacticacid,0.883979,0.106153,0.009868,0.488036,0.492992,0.018972,Forced Convergence in LIQ_EX - results might n...
934,40.0,0.9543,0,0.0457,0,1.0,0,h2o,butanone,lacticacid,0.872537,0.113323,0.014140,0.504666,0.469439,0.025895,Forced Convergence in LIQ_EX - results might n...
935,40.0,0.9407,0,0.0593,0,1.0,0,h2o,butanone,lacticacid,0.573456,0.397547,0.028997,0.571822,0.399169,0.029009,Forced Convergence in LIQ_EX - results might n...


In [11]:
from COSMOtherm.src.Chemfuncs import calc_la_molefrac

tC_range = [20, 30, 40]  # [°C]
massconcentration_lacticacid_range = [5, 10, 20, 50, 100, 150, 200, 250]
x1_lacticacid_range = calc_la_molefrac(
        massconcentration_lacticacid=massconcentration_lacticacid_range
        )


In [ ]:


# Select only the relevant columns as a subspace of warnings_df
subspace = failed_convergence_df[['Temperature (°C)', 'Phase 1 x(3) input', 'Compound 2']]
subspace = subspace.rename(columns={
    'Temperature (°C)': 'temperature',
    'Phase 1 x(3) input': 'x1_lacticacid',
    'Compound 2': 'COSMO_name'
})


q:\Groups\eicr students\Stefan Tönnis\GIT_Repos\BayesianThompsonSamplingOptimization\COSMOtherm\src\funcs.py:51: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  solvents = pd.read_csv(solvents_fullpath, delim_whitespace=True)


In [13]:
from COSMOtherm.src.Design_Matrix import build_design_matrix


design_matrix = build_design_matrix(
            tC_range=tC_range, 
            x1_lacticacid_range=x1_lacticacid_range, 
            solvents=solvents,
            reduction=0
            )

In [ ]:
design_matrix

In [ ]:
# Perform a left join to add the SMILES column
solvents = load_and_sort_solvents(
    Config.solvents_fullpath
    )

output_data = output_data.merge(
    solvents[['COSMO_name', 'SMILES']],
    how='left',
    left_on='Compound 2',
    right_on='COSMO_name'
)
